In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
import numpy as np
import re
import seaborn as sb
import io
import os
import pandas as pd
import fnmatch
from numpy import percentile
from scipy import stats
import glob
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.multitest import multipletests


In [ ]:
def read_txt2(path):
    vcf_df = pd.read_csv(path, sep='\t', comment='#', header=None)
    lst=vcf_df[7].str.split(';')
    fxhold = [int(row) for row in vcf_df[1]]
    fhold = [float(row[1].split('=')[1]) for row in lst]
    refhold = [row for row in vcf_df[3]]
    althold = [row for row in vcf_df[4]]
    return fhold, fxhold, refhold, althold
def get_rRNA_bounds(gb_file_path):
    gb_file = open(gb_file_path)
    pattern_1 = r'^\s+rRNA'
    pattern_2 = r'\d+'
    points = []
    for line in gb_file:
        if re.search(pattern_1, line):
            send = re.findall(pattern_2, line)
            points.append([int(send[0]), int(send[1])])
    return points
def find_vcf_directories(root_dir):
    vcf_directories = []
    for dirpath, dirnames, filenames in os.walk(root_dir):
        for filename in filenames:
            if filename.endswith('_rDNA.vcf'):
            # if filename.endswith('_rDNA.vcf_temp7.txt'):
                vcf_directories.append(os.path.join(dirpath, filename))
    return vcf_directories

In [ ]:
####################regional information that we know of########################################
########################################################################################################################
#28S ExpansionSegments
df1
#18S ExpansionSegments
df2
#homopolymers
df3
#conserved regions
df4
#pseudogene regions
df6
#rRNA regions(18S, 5.8S, 28S)
gb_file_path='U13369.1_Human_rDNA_repeat_unit.gb'
rRNA_positions = get_rRNA_bounds(gb_file_path)
zoomed_range = [0,13314]

In [ ]:
expansion_positions1=[]
for row in df1.iterrows():
    hold=[]
    hold.append(row[1]['Start'])
    hold.append(row[1]['End'])
    expansion_positions1.append(hold)
# print(expansion_positions1)
expansion_positions2=[]
for row in df2.iterrows():
    hold=[]
    hold.append(row[1]['Start'])
    hold.append(row[1]['End'])
    expansion_positions2.append(hold)
# print(expansion_positions2)
hmplr_pos=[]
for row in df3.iterrows():
    hold=[]
    hold.append(row[1]['Start'])
    hold.append(row[1]['End'])
    hmplr_pos.append(hold)
# print(hmplr_pos)
csvd_pos=[]
for row in df4.iterrows():
    hold=[]
    hold.append(row[1]['Start'])
    hold.append(row[1]['End'])
    csvd_pos.append(hold)


# Flatten each region into sets of positions
overlap=[10809, 10810, 10811, 10812, 10813, 10814]
expansion_28 = set(i for start, end in expansion_positions1 for i in range(start, end + 1))
csvd_28 = set(i for start, end in csvd_pos for i in range(start, end + 1))
csvd_28 = csvd_28 -set(overlap)
used = expansion_28.union(csvd_28)
full_range=[7935, 12969]
hmply= set(i for start, end in hmplr_pos for i in range(start, end + 1))
# pseudo= set(i for start, end in pseud_pos for i in range(start, end + 1))
# All positions in full range
all_positions = set(range(full_range[0], full_range[1] + 1))

# The "rest" are positions not in either of the two sets
rest_28 = all_positions - used

print("expansion:", len(expansion_28))
print("csvd:", len(csvd_28))
print("rest:", len(rest_28))
print("28s:", len(rest_28)+len(csvd_28)+len(expansion_28))

In [ ]:
root_directory2 = 
vcf_directories = find_vcf_directories(root_directory2)

In [ ]:
done2={}
for file in vcf_directories:
    sdf= pd.DataFrame(columns=['x', 'y', 'ref', 'alt'])
    hold,xhold,rh,ah=read_txt2(file)
    data = pd.DataFrame({'x': xhold, 'y': hold, 'ref':rh, 'alt': ah})
    for i, segment in data.iterrows():
        xtmp, alt, ref, tmp  = int(segment['x']), segment['alt'],segment['ref'],  segment['y']
        if (xtmp,ref,alt) in done2:
            done2[(xtmp,ref,alt)].append(tmp)
        else:
            done2[(xtmp,ref,alt)] = []
            done2[(xtmp,ref,alt)].append(tmp)

In [ ]:
plt.figure(figsize=(12, 6))

for i, (start, end) in enumerate(rRNA_positions):

    plt.axvline(x=start, color='orange', linestyle='--', alpha=0.7)
    plt.axvline(x=end, color='orange', linestyle='--', alpha=0.7)
    rRNA_region = plt.axvspan(start, end, alpha=0.25, color='lightgreen')
xhold=[]
yhold=[]
for _, row in allele_of_interest.iterrows():
    for entry in row['values']:
        xhold.append(row['key'][0])
        yhold.append(entry)

plt.scatter(xhold,yhold, s=0.7, color='Blue')

plt.xlabel('Position')
plt.ylabel('Allele Frequency')
plt.title("Variantion of Allele Frequence in the rRNA region")
plt.xlim(zoomed_range)
plt.ylim(0,1.01)
plt.tight_layout()


In [ ]:
colorl=[0]*13314
############overall vairant distribution of the entire cohort########################
plt.figure(figsize=(12, 6))
for i, (start, end) in enumerate(pseud_pos):
    # rRNA_region = plt.axvspan(start, end, 0, 0.3, alpha=0.25, color='blue')
    rRNA_region = plt.axvspan(start, end, alpha=0.2, color='grey')

for i, (start, end) in enumerate(rRNA_positions):

    plt.axvline(x=start, color='orange', linestyle='--', alpha=0.7)
    plt.axvline(x=end, color='orange', linestyle='--', alpha=0.7)
    rRNA_region = plt.axvspan(start, end, alpha=0.25, color='lightgreen')
for i, (start, end) in enumerate(expansion_positions2):
    #print(row[1]['Start'])
#    plt.axvline(x=row[1]['Start'], color='grey', linestyle='--', alpha=0.1)
#    plt.axvline(x=row[1]['End'], color='grey', linestyle='--', alpha=0.1)
    rRNA_region = plt.axvspan(start, end, alpha=0.2, color='grey')
    
for i, (start, end) in enumerate(expansion_positions1):
#    plt.axvline(x=row[1]['Start'], color='grey', linestyle='--', alpha=0.1)
#    plt.axvline(x=row[1]['End'], color='grey', linestyle='--', alpha=0.1)
    rRNA_region = plt.axvspan(start, end, alpha=0.2, color='grey')

for i, (start, end) in enumerate(hmplr_pos):
    rRNA_region = plt.axvspan(start, end, alpha=0.2, color='grey')


for file in vcf_directories:
# for file in range(10):
    hold,xhold,rh,ah=read_txt2(file)

    greyx=[]
    greyy=[]
    starx=[]
    stary=[]
    # Convert xhold and hold to DataFrame for easier manipulation
    data = pd.DataFrame({'x': xhold, 'y': hold})
    # Loop through each segment in df1(28S)
    for _, segment in df1.iterrows():
        start, end = segment['Start'], segment['End']
        
        # Select rows where 'x' is within the segment range
        mask = (data['x'] >= start) & (data['x'] <= end)
        selected_data = data[mask]
        
        # Append selected x and y to greyx and greyy
        greyx.extend(selected_data['x'].tolist())
        greyy.extend(selected_data['y'].tolist())
        
        # Remove the selected rows from data
        data = data[~mask]
    # Loop through each segment in df2(18S)
    for _, segment in df2.iterrows():
        start, end = segment['Start'], segment['End']
        
        # Select rows where 'x' is within the segment range
        mask2 = (data['x'] >= start) & (data['x'] <= end)
        selected_data2 = data[mask2]
        
        # Append selected x and y to greyx and greyy
        greyx.extend(selected_data2['x'].tolist())
        greyy.extend(selected_data2['y'].tolist())
        
        # Remove the selected rows from data
        data = data[~mask2]
    # Loop through each segment in df3(homopolymers)
    for _, segment in df3.iterrows():
        start, end = segment['Start'], segment['End']
        
        # Select rows where 'x' is within the segment range
        mask3 = (data['x'] >= start) & (data['x'] <= end)
        selected_data3 = data[mask3]
        
        # Append selected x and y to greyx and greyy
        greyx.extend(selected_data3['x'].tolist())
        greyy.extend(selected_data3['y'].tolist())
        
        # Remove the selected rows from data
        data = data[~mask3]
    for _, segment in df6.iterrows():
        start, end = segment['Start'], segment['End']
        
        # Select rows where 'x' is within the segment range
        mask6 = (data['x'] >= start) & (data['x'] <= end)
        selected_data6 = data[mask6]
        
        # Append selected x and y to greyx and greyy
        greyx.extend(selected_data6['x'].tolist())
        greyy.extend(selected_data6['y'].tolist())
        
        # Remove the selected rows from data
        data = data[~mask6]

    # Loop through each segment in df4(conserved)
    for _, segment in df4.iterrows():
        start, end = segment['Start'], segment['End']
        
        # Select rows where 'x' is within the segment range
        mask4 = (data['x'] >= start) & (data['x'] <= end)
        selected_data4 = data[mask4]
        
        # Append selected x and y to greyx and greyy
        starx.extend(selected_data4['x'].tolist())
        stary.extend(selected_data4['y'].tolist())
        
        # Remove the selected rows from data
        data = data[~mask4]

    
    for i, segment in data.iterrows():
        xtmp, tmp = int(segment['x']), segment['y']
        colorl[xtmp]=colorl[xtmp]+1


    xhold=data['x']
    hold=data['y']
    plt.scatter(greyx,greyy, s=0.7, color='Gray')
    plt.scatter(xhold,hold, s=0.7, color='Blue')
    plt.scatter(starx,stary, s=3, color='Red',marker='*')
    


plt.xlabel('Position')
plt.ylabel('Allele Frequency')
plt.title("Variantion of Allele Frequence in the rRNA region")

plt.xlim(zoomed_range)
plt.ylim(0,1.01)
plt.tight_layout()

In [ ]:
#### trio geographics
runid_df=pd.read_csv('1000data.csv').rename(columns={'Sample Name':'SampleID'})
familyid_df=pd.read_csv('20130606_g1k_3202_samples_ped_population.txt',sep=' ')
merged_df=pd.merge(familyid_df, runid_df,on='SampleID')
child_ids = set(merged_df[
    (merged_df['FatherID'] != '0') | (merged_df['MotherID'] != '0')
]['Run'])
vcf_directories_nochild=[i for i in vcf_directories if i.split('/')[-2] not in child_ids]
vcf_directories_nochild_id=[i.split('/')[-2] for i in vcf_directories_nochild]
merged_df=merged_df[merged_df['Run'] .isin(vcf_directories_nochild_id)]
all_all={}
for file in vcf_directories_nochild:
    hold, xhold, rh, ah = read_txt2(file)
    name_id=file.split('/')[-2]
    # Create DataFrame for this file
    df = pd.DataFrame({'x': xhold, 'y': hold, 'ref':rh, 'alt':ah})
    for i, row in df.iterrows():
        if (row['x'],row['ref'], row['alt']) not in all_all:
            all_all[(row['x'],row['ref'], row['alt'])]=[(name_id,row['y'])]
            # all_all[(row['x'],row['ref'], row['alt'])]=[name_id]
        else:
            all_all[(row['x'],row['ref'], row['alt'])].append((name_id,row['y']))
            # all_all[(row['x'],row['ref'], row['alt'])].append(row['y'])
all_list={}
for key, value in all_all.items():
    if (key[0] not in hmply) and (str(key) not in list(pseudo['ID'])) :
        all_list[key]=value

In [ ]:
###shareness analysis
minor_allele={}
for key,value in all_list.items():
    if len(vcf_directories_nochild)*0.01>len(value)>1:
        # print(len(value))
        minor_allele[key]=value

minor_ale_population={}
for key,value in minor_allele.items():
    new_value=[]
    for item in value:
        # new_value.append(merged_df[merged_df['Run']==item]['Population'].values)
        new_value.append((merged_df[merged_df['Run']==item[0]]['Population'].values[0],item[1]))
    minor_ale_population[key]=new_value


median_allele={}
for key,value in all_list.items():
    if len(vcf_directories_nochild)*0.05>=len(value)>=len(vcf_directories_nochild)*0.01:
        # print(len(value))
        median_allele[key]=value
median_ale_population={}
for key,value in median_allele.items():
    new_value=[]
    for item in value:
        # new_value.append(merged_df[merged_df['Run']==item]['Population'].values)
        new_value.append((merged_df[merged_df['Run']==item[0]]['Population'].values[0],item[1]))
    median_ale_population[key]=new_value

    
pop_counts = merged_df['population'].value_counts().to_dict()
total_samples = sum(pop_counts.values())
data_for_plotting = []

for allele, pops in minor_ale_population.items():
    flat_list = [item[0] for item in pops]
    enrichment = calculate_enrichment_pvalue(flat_list, pop_counts)
    
    # Find the most significant population for this allele
    # We take the minimum p-value among all populations present for this allele
    pops_involved = enrichment.keys()
    if not pops_involved:
        continue
        
    best_pop = min(pops_involved, key=lambda x: enrichment[x]['p_value'])
    min_p = enrichment[best_pop]['p_value']
    
    # Convert to -log10(p)
    neg_log_p = -np.log10(min_p + 1e-300) 
    
    data_for_plotting.append({
        'Allele': f"{allele[0]}_{allele[1]}/{allele[2]}",
        'NegLogP': neg_log_p,
        'p':min_p,
        'PrimaryPop': best_pop,
        'Count': len(flat_list)
    })

df_enrichment = pd.DataFrame(data_for_plotting)
df_enrichment
#### BH
# 1. Ensure your DataFrame is created as before
df_enrichment = pd.DataFrame(data_for_plotting)

# 2. Apply the Benjamini-Hochberg correction
# We extract the 'p' column and apply the 'fdr_bh' method
# alpha=0.05 is the target FDR threshold
reject, q_values, alphacSidak, alphacBonf = multipletests(
    df_enrichment['p'], 
    alpha=0.05, 
    method='fdr_bh'
)

# 3. Add the adjusted p-values (q-values) and the significance decision back to the DF
df_enrichment['q_value'] = q_values
df_enrichment['is_significant'] = reject

# 4. Calculate -log10(q-value) for plotting (Manhattan or Volcano plots)
df_enrichment['NegLogQ'] = -np.log10(df_enrichment['q_value'] + 1e-300)

# 5. (Optional) Sort by significance
df_enrichment = df_enrichment.sort_values('q_value')

print(f"Significant hits after BH correction: {df_enrichment['is_significant'].sum()}")
rows = []

for count in sorted(df_enrichment['Count'].unique()):
    current = df_enrichment[df_enrichment['Count'] == count]
    
    n_total = len(current)
    # n_sig = len(current[current['p'] < 0.05])
    n_sig = current['is_significant'].sum()
    
    rows.append({
        "Count": count,
        "Significant": n_sig,
        "Not Significant": n_total - n_sig
    })

df_plot = pd.DataFrame(rows).set_index("Count")

df_plot.plot(
    kind="bar",
    stacked=True,
    color=["#4c72b0", "#dd8452"]
)

plt.ylabel("Number of variants")
plt.title("Minor allele distribution")
plt.legend(title="p-value")
plt.tight_layout()
plt.show()


rows = []

for count in sorted(df_enrichment['Count'].unique()):
    current = df_enrichment[df_enrichment['Count'] == count]
    
    n_total = len(current)
    # n_sig = len(current[current['p'] < 0.05])
    n_sig = current['is_significant'].sum()
    n_nonsig = n_total - n_sig
    
    rows.append({
        "Count": count,
        "Significant": n_sig / n_total if n_total > 0 else 0,
        "Not Significant": n_nonsig / n_total if n_total > 0 else 0
    })

df_plot = pd.DataFrame(rows).set_index("Count")

df_plot.plot(
    kind="bar",
    stacked=True,
    colormap="viridis"
)

plt.ylabel("Proportion")
plt.title("Minor allele distribution")
plt.legend(title="p-value")
plt.tight_layout()
plt.show()

In [ ]:
### shareness overview
bigdf= pd.DataFrame(list(all_list.items()), columns=['key', 'values'])
hist=[]
c=0
for _, row in bigdf.iterrows():
    huh=len(row['values'])
    hist.append(huh)


# Create the larger plot
fig, ax1 = plt.subplots(figsize=(12, 4))
ax1.hist(hist,bins=300)
ax1.set_xlabel("# of samples shared across")
ax1.set_ylabel("# of variants")


##############
ax1.set_ylim(0,1000)
ax1.set_xlim(-10,3200)
ax1.set_yticks([])

ax2 = plt.axes([0.2, 0.5, 0.15, 0.3])  
ax2.hist(hist,bins=3190)
ax2.set_xlim(1, 11)
ax2.set_yticks([0,hist.count(1)+10])
ax2.set_xticks([1,2,3,4,5,6,7,8,9,10])


ax3 = plt.axes([0.65, 0.5, 0.15, 0.3]) 
ax3.hist(hist,bins=3190)
ax3.set_xlim(3170,3190)
ax3.set_ylim(0,hist.count(3190)+10)
ax3.set_yticks([0,hist.count(3190)])

plt.savefig("Figure2-a_no_pseudo.svg")

In [ ]:
##copy number analysis


# 1. Define the total size of the dataframe (0 to 13314 is 13315 elements)
total_length = 13314
all_indices = np.arange(total_length)

# 2. Define the ranges for Mask 1
ranges = [[3657, 5527], [6623, 6779], [7935, 12969]]

# 3. Create Mask 1 (initialize as False, then fill in the ranges)
mask1_array = np.zeros(total_length, dtype=bool)

for start, end in ranges:
    # We add +1 to 'end' to make it inclusive [start, end]
    mask1_array[start : end + 1] = True

# 4. Create Mask 2 (Logical NOT of Mask 1)
mask2_array = ~mask1_array

# --- Verification ---
print(f"Total length: {total_length}")
print(f"Count in Mask 1 (Inside ranges): {np.sum(mask1_array)}")
print(f"Count in Mask 2 (Outside ranges): {np.sum(mask2_array)}")

# 3099706404 fiona
#3298912062. 3137300923 https://www.ncbi.nlm.nih.gov/grc/human/data
bases=pd.read_csv('1000data.csv.csv',header=1)
hold=[]
new_bases=bases[['Run','Bases']]
for i, row in new_bases.iterrows():
    hold.append(row['Bases']/3137300923)
new_bases['Genome_coverage']=hold

root_directory2 = ''
def find_coverage_directories(root_dir):
    vcf_directories = []
    for dirpath, dirnames, filenames in os.walk(root_dir):
        for filename in filenames:
            if filename.endswith('_coverage.txt'):
            # if filename.endswith('_rDNA.vcf_temp7.txt'):
                vcf_directories.append(os.path.join(dirpath, filename))
    return vcf_directories
coverage_directories = find_coverage_directories(root_directory2)
results=[]
for filename in coverage_directories:
    # if filename.endswith("_rDNA_coverage.txt"):
        # file_path = os.path.join(directory_path, filename)
    try:
        df = pd.read_csv(filename, sep='\t', header=None, names=["Human", "Column2", "Coverage"])
        coding_df=df[mask1_array]
        spacer_df=df[mask2_array]
        coding_mean_coverage = coding_df["Coverage"].mean()
        spacer_mean_coverage = spacer_df["Coverage"].mean()
    
        run_id = filename.split('/')[-2]
    
        results.append({"Run": run_id, "rRNA_avg": coding_mean_coverage,"spacer_avg":spacer_mean_coverage})
    except:
        print(run_id)
        

results_df = pd.DataFrame(results)
merge_bases=pd.merge(new_bases,results_df,on='Run', how='left')

merge_bases['rRNA_copy_number']=merge_bases['rRNA_avg']/merge_bases['Genome_coverage']
merge_bases['spacer_copy_number']=merge_bases['spacer_avg']/merge_bases['Genome_coverage']
plt.figure(figsize=(10, 6))
plt.hist(merge_bases['rRNA_copy_number'], bins=50, alpha=0.5, label='rRNA', color='blue')
plt.hist(merge_bases['spacer_copy_number'], bins=50, alpha=0.5, label='Spacer', color='red')
plt.title("Distribution of Copy Numbers")
plt.xlabel("Copy Number ")
plt.ylabel("Frequency")
plt.legend()
plt.show()

plt.figure(figsize=(10, 4))
sns.boxplot(x=merge_bases['rRNA_copy_number'])
plt.xlabel('Estimated copy number by rRNA region')
plt.show()



plt.figure(figsize=(12, 6))

# 1. Your original data
x = merge_bases_2['rRNA_copy_number']
y = merge_bases_2['#_unique_allele']
plt.scatter(x, y, alpha=0.5, label='Data points')

slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
r_squared = r_value**2  # Calculate the coefficient of determination (R^2)


line_x = np.linspace(x.min(), x.max(), 100)
line_y = slope * line_x + intercept

# 4. Plot the line
plt.plot(line_x, line_y, color='red', linewidth=2, 
         label=f'Fit: $y = {slope:.2f}x + {intercept:.2f}$')


stats_text = (f"$r = {r_value:.3f}$\n"
              f"$R^2 = {r_squared:.3f}$\n"
              f"$P = {p_value:.3e}$")


plt.gca().text(0.05, 0.95, stats_text, transform=plt.gca().transAxes, 
               fontsize=12, verticalalignment='top', 
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# 6. Formatting
plt.xlabel("Estimated copy number by rRNA region")
plt.ylabel("Number of Unique Alleles")
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()


plt.figure(figsize=(12, 6))
x = merge_bases_3['coding_copy_number']
y = merge_bases_3['median_ivf']
plt.scatter(x, y)


slope, intercept = np.polyfit(x, y, 1)

line_x = np.linspace(x.min(), x.max(), 100)
line_y = slope * line_x + intercept

# 4. Plot the line
plt.plot(line_x, line_y, color='red', label=f'Fit: y={slope:.4f}x + {intercept:.2f}')
plt.xlabel("Estimated copy number by coding region")       # X-axis label
plt.ylabel("median_ivf")
plt.legend()
plt.show()

In [ ]:
### coverage overview
coverage=pd.read_csv('/projects/rps/cgsb/hochwagen/Human_rDNA_project/expreads/ERR3239320/ERR3239320_rDNA_coverage.txt',sep='\t',header=None)
plt.figure(figsize=(12, 6))
plt.scatter(coverage[1], np.log10(coverage[2]), s=5, color='Blue')

# Add vertical lines and shaded regions
for start, end in rRNA_positions:
    plt.axvline(x=start, color='orange', linestyle='--', alpha=0.7)
    plt.axvline(x=end, color='orange', linestyle='--', alpha=0.7)
    plt.axvspan(start, end, alpha=0.25, color='lightgreen')


plt.xlabel('Position')
plt.ylabel('Log Coverge')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
### 28S analysis 
CON=[]
EXP=[]
RST=[]
for file in vcf_directories:
    hold, xhold, rh, ah = read_txt2(file)

    # Create DataFrame for this file
    df = pd.DataFrame({'x': xhold, 'y': hold})
    # df = df[(df['x'] >= 7935) & (df['x'] <= 12969)]
    one = df[df['x'].isin(csvd_28 )]['y']
    two = df[df['x'].isin(expansion_28)]['y']
    thr = df[df['x'].isin(rest_28)]['y']
    CON.append(one)
    EXP.append(two)
    RST.append(thr)
data = [EXP,CON, RST]
flierprops = {'marker': 'o', 'markersize': 2}
colors = ['lightgrey', 'red', 'lightgreen']  # Adjust color order as needed

fig = plt.figure(figsize =(12, 6))
ax = fig.add_axes([0, 0, 1, 1])
bp = ax.boxplot(data, vert=True, patch_artist=True,labels=["Expansion Segments",  "Conserved Regions",'Rest of 28S'], flierprops=flierprops, showmeans=True,meanline=True)

for box, median, label in zip(bp['boxes'], bp['medians'], colors):
    box.set_facecolor(label)
for line in bp['medians']:
    x, y = line.get_xydata()[1]
    plt.text(x+0.1, y, '%.0f' % y,
         horizontalalignment='center') 

plt.xlabel("Segments")
plt.ylabel("Allele Count")

plt.title("Variantion of Allele Counts by Regions in 28S")
plt.show()

import matplotlib.ticker as mtick
CON=[]
EXP=[]
RST=[]
for key, value in all_list.items():
   # print(key[0]) 

    if (key[0] in csvd_28):
        CON.append(np.median(value))
    elif (key[0] in expansion_28):
        EXP.append(np.median(value))
    elif (key[0] in rest_28) :
        RST.append(np.median(value))
data = [EXP, CON, RST]
flierprops = {'marker': 'o', 'markersize': 2}
colors = ['lightgrey', 'lightgreen', 'red']  # Adjust color order as needed


fig, ax = plt.subplots(figsize =(12, 6))

bp = ax.boxplot(data, vert=True, patch_artist=True,labels=["Expansion Segments", "Conserved Regions", 'Rest of 28S'], flierprops=flierprops)

for box, median, label in zip(bp['boxes'], bp['medians'], colors):
    box.set_facecolor(label)
for line in bp['medians']:
    x, y = line.get_xydata()[1] 
    plt.text(x+0.1, y, f'{y * 100:.2f}%',
         horizontalalignment='center') 

plt.xlabel("Segments")
plt.ylabel("Median Allele Frequency")
# plt.title("Boxplot of Data")
plt.title("Variantion of Allele Frequency by Regions in 28S")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
plt.tight_layout()
plt.show()    

## indel
indel_len={}
for file in vcf_directories:
    hold,xhold,rh,ah=read_txt2(file)
    df = pd.DataFrame({'x': xhold, 'y': hold, 'ref':rh, 'alt':ah})
    for i, row in df.iterrows():
        if len(row['ref'])!=len(row['alt']):
            indel_len[(row['x'],row['ref'],row['alt'])]=abs(len(row['ref'])-len(row['alt']))
all_list={}
for key, value in indel_len.items():
    # print(key)
    if (key[0] not in hmply) and (key not in name) :
        all_list[key]=value

CON=[]
EXP=[]
RST=[]
for key, value in all_list.items():
   # print(key[0]) 
    # if key[0] not in pseudo and key[0] not in hmply:
    if key[0] in csvd_28:
        CON.append(value)
    if key[0] in expansion_28:
        EXP.append(value)
    if key[0] in rest_28:
        RST.append(value)
data = [EXP,CON,RST]

In [ ]:
############ vairant distribution by segment########################
all_positions=[[1,3656,"5'ETS"],[3657, 5527,'18S'],[5528,6622,'ITS1'], [6623, 6779,'5.8S'], [6780,7934,'ITS2'],[7935, 12969,'28S'],[12970,13314,"3'ETS"]]
labels=["5'ETS", '18S', 'ITS1', '5.8S', 'ITS2', '28S', "3'ETS"]
ETS1x, ETS2x, ITS1x, ITS2x, etnSx, fivSx, ttySx, ETS1y, ETS2y, ITS1y, ITS2y, etnSy, fivSy, ttySy = [[] for _ in range(14)]


for file in vcf_directories:
    hold,xhold,rh,ah=read_txt2(file)
    # Convert xhold and hold to DataFrame for easier manipulation
    data = pd.DataFrame({'x': xhold, 'y': hold})
    # Loop through each segment in df1(28S)
    for pair in all_positions:
        start, end, name = pair[0], pair[1], pair[2]
        # print(start, end, name)
        
        # Select rows where 'x' is within the segment range
        mask = (data['x'] >= start) & (data['x'] <= end) & (data['x'] not in (hmply and pseudo))
        selected_data = data[mask]
        if name == "5'ETS" :
            ETS1x.extend(selected_data['x'].tolist())
            ETS1y.extend(selected_data['y'].tolist())
        if name == "18S" :
            etnSx.extend(selected_data['x'].tolist())
            etnSy.extend(selected_data['y'].tolist())
        if name == "ITS1" :
            ITS1x.extend(selected_data['x'].tolist())
            ITS1y.extend(selected_data['y'].tolist())
        if name == "5.8S" :
            fivSx.extend(selected_data['x'].tolist())
            fivSy.extend(selected_data['y'].tolist())
        if name == "ITS2" :
            ITS2x.extend(selected_data['x'].tolist())
            ITS2y.extend(selected_data['y'].tolist())
        if name == "28S" :
            ttySx.extend(selected_data['x'].tolist())
            ttySy.extend(selected_data['y'].tolist())
        if name == "3'ETS" :
            ETS2x.extend(selected_data['x'].tolist())
            ETS2y.extend(selected_data['y'].tolist())

# Create a sample DataFrame
# data = {'col1': [1, 5, 8, 3, 7, 2, 9, 4, 6],'col2': [1, 5, 8, 3, 7, 2, 9, 4, 6]}
data = [ETS1y, etnSy, ITS1y, fivSy, ITS2y, ttySy, ETS2y]

flierprops = {'marker': 'o', 'markersize': 2}
# Define boxplot colors
colors = ['lightgrey', 'lightgreen', 'lightgrey', 'lightgreen', 'lightgrey', 'lightgreen', 'lightgrey']  # Adjust color order as needed

# fig = plt.figure(figsize =(12, 6))
fig, ax = plt.subplots(figsize=(12, 6))
# Creating axes instance
# ax = fig.add_axes([0, 0, 1, 1])
# Add labels and title (optional)
bp = ax.boxplot(data, vert=True, patch_artist=True,labels=["5'ETS", '18S', 'ITS1', '5.8S', 'ITS2', '28S', "3'ETS"]
                , flierprops=dict(marker='.', markersize=0.5)
                , showmeans=True,meanline=True
               )
#notch=True, vert=True,
# Color boxes and medians based on labels
for box, median, label in zip(bp['boxes'], bp['medians'], colors):
    box.set_facecolor(label)
    # median.set_color(label)

for line in bp['medians']:
    # get position data for median line
    x, y = line.get_xydata()[1] # top of median line
    # overlay median value
    plt.text(x, y, '%.3f' % y,
         horizontalalignment='center') # draw above, centered

plt.xlabel("Segments")
plt.ylabel("Allele Frequency")
plt.title("Variantion of Allele Frequencies by Segment")
plt.tight_layout()
plt.show()



############ median distribution by segment########################
all_positions=[[1,3656,"5'ETS"],[3657, 5527,'18S'],[5528,6622,'ITS1'], [6623, 6779,'5.8S'], [6780,7934,'ITS2'],[7935, 12969,'28S'],[12970,13314,"3'ETS"]]
labels=["5'ETS", '18S', 'ITS1', '5.8S', 'ITS2', '28S', "3'ETS"]
ETS1x, ETS2x, ITS1x, ITS2x, etnSx, fivSx, ttySx, ETS1y, ETS2y, ITS1y, ITS2y, etnSy, fivSy, ttySy = [[] for _ in range(14)]


for key, value in all_list.items():
    y_values = value
    if all_positions[0][0]<=key[0]<=all_positions[0][1]:
        ETS1y.append(np.median(y_values))
        ETS1x.append(np.median(key[0]))
    elif all_positions[1][0]<=key[0]<=all_positions[1][1]:
        etnSy.append(np.median(y_values))
        etnSx.append(np.median(key[0]))
    elif all_positions[2][0]<=key[0]<=all_positions[2][1]:
        ITS1y.append(np.median(y_values))
        ITS1x.append(np.median(key[0]))
    elif all_positions[3][0]<=key[0]<=all_positions[3][1]:
        fivSy.append(np.median(y_values))
        fivSx.append(np.median(key[0]))
    elif all_positions[4][0]<=key[0]<=all_positions[4][1]:
        ITS2y.append(np.median(y_values))
        ITS2x.append(np.median(key[0]))
    elif all_positions[5][0]<=key[0]<=all_positions[5][1]:
        ttySy.append(np.median(y_values))
        ttySx.append(np.median(key[0]))
    elif all_positions[6][0]<=key[0]<=all_positions[6][1]:
        ETS2y.append(np.median(y_values))
        ETS2x.append(np.median(key[0]))

data = [ETS1y, etnSy, ITS1y, fivSy, ITS2y, ttySy, ETS2y]
data1 = [ETS1x, etnSx, ITS1x, fivSx, ITS2x, ttySx, ETS2x]
n_values = [len(d) for d in data] 
unique_positions = [len(np.unique(d)) for d in data1]

flierprops = {'marker': 'o', 'markersize': 2}
# Define boxplot colors
colors = ['lightgrey', 'lightgreen', 'lightgrey', 'lightgreen', 'lightgrey', 'lightgreen', 'lightgrey']  # Adjust color order as needed

fig, ax = plt.subplots(figsize=(12, 6))
bp = ax.boxplot(data, vert=True, patch_artist=True,labels=["5'ETS", '18S', 'ITS1', '5.8S', 'ITS2', '28S', "3'ETS"]
                , flierprops=dict(marker='.', markersize=1)
              
               )

for i, n in enumerate(n_values):

    x_pos = i + 1
    y_pos = 1.05
    ax.text(x_pos, y_pos, f'N/Pos={n}/{unique_positions[i]}', ha='center', va='bottom', fontsize=8)

for box, median, label in zip(bp['boxes'], bp['medians'], colors):
    box.set_facecolor(label)
    # median.set_color(label)

for line in bp['medians']:
    # get position data for median line
    x, y = line.get_xydata()[1] # top of median line
    # overlay median value
    plt.text(x, y, '%.3f' % y,
         horizontalalignment='center') # draw above, centered

plt.xlabel("Segments")
plt.ylabel("Allele Frequency")

ax.set_title("Variantion of Allele Frequencies by Segment", y=1.07) 
plt.tight_layout()
plt.show()




In [ ]:
####nucleotide composition

def extract_sequence_from_genbank(gb_file_path):
    sequence_lines = []
    with open(gb_file_path, 'r') as gb_file:
        recording = False
        for line in gb_file:
            if line.strip().startswith('ORIGIN'):
                recording = True
                continue
            if recording:
                if line.strip().startswith('//'):  # End of the sequence
                    break
                # Remove line numbers and spaces, keep only bases
                clean_line = ''.join(re.findall(r'[acgtACGT]+', line))
                sequence_lines.append(clean_line)
    return ''.join(sequence_lines).upper()  # Concatenate and uppercase
sequence = extract_sequence_from_genbank(gb_file_path)
position_lists = [expansion_28, csvd_28, rest_28]
for idx, positions in enumerate(position_lists, start=1):
    # Collect bases at those positions
    bases = [sequence[pos-1] for pos in positions]  # 1-based to 0-based
    count = Counter(bases)
    print(f"List {idx} counts: {dict(count)} {len(positions)}" )

In [ ]:
##synthetic simulation
p_path7000=
p_path30000=
p_path50000=
p_path100000=
p_path200000=
d_7000=[]
df_p=read_vcf(p_path7000)
for i in range(len(df_p)):
    huh=df_p['INFO'].str.split(';')[i][1].split('=')[1]
    d_7000.append(float(huh))
# print(d_7000)
d_30000=[]
df_p=read_vcf(p_path30000)
for i in range(len(df_p)):
    huh=df_p['INFO'].str.split(';')[i][1].split('=')[1]
    d_30000.append(float(huh))
# print(d_30000)
d_50000=[]
df_p=read_vcf(p_path50000)
for i in range(len(df_p)):
    huh=df_p['INFO'].str.split(';')[i][1].split('=')[1]
    d_50000.append(float(huh))
# print(d_50000)
d_100000=[]
df_p=read_vcf(p_path100000)
for i in range(len(df_p)):
    huh=df_p['INFO'].str.split(';')[i][1].split('=')[1]
    d_100000.append(float(huh))
# print(d_100000)
d_200000=[]
df_p=read_vcf(p_path200000)
for i in range(len(df_p)):
    huh=df_p['INFO'].str.split(';')[i][1].split('=')[1]
    d_200000.append(float(huh))
# print(d_200000)
box=[]
box.append(d_7000)
box.append(d_30000)
box.append(d_50000)
box.append(d_100000)
box.append(d_200000)
import seaborn as sns
data = pd.DataFrame({'Allele Frequency': np.concatenate(box),
                     'Coverage': np.repeat(['7X', '30X', '50X', '100X', '200X'],
                                           [len(d) for d in box])})

plt.figure(figsize=(10, 6))
sns.boxplot(x='Coverage', y='Allele Frequency', data=data)
sns.stripplot(x='Coverage', y='Allele Frequency', data=data, color='black', alpha=0.5, jitter=True)
plt.show()